# RL2AC Parameter-Sweep Evaluation Analysis

This notebook is adapted from the multi-terrain evaluation analysis notebook, but changes the aggregation level from

```text
approach × terrain × disturbance_condition
```

to

```text
approach × terrain × disturbance_condition × alpha × kappa × lambda_0 × k_0
```

Expected RL2AC filename pattern:

```text
{task}_{terrain}_{disturbance_condition}_alpha_{alpha}_kappa_{kappa}_lambda0_{lambda_0}_k0_{k_0}.csv
```

For example:

```text
go1_rl2ac_rough_payload_alpha_10_kappa_0.1_lambda0_0.3_k0_5.csv
```

The parser also accepts `lambda_0` and `k_0` spellings in filenames.

In [ ]:
import os
import re
import ast
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

## Configuration

In [ ]:
# --- User settings ---
EXP_FOLDER = "exp_data_corl_rl2ac/sample_param_search"
APPROACH = "go1_rl2ac"

# Set to None to load every terrain / disturbance condition.
# Otherwise use strings like "rough", "stairs", "plane", etc.
TERRAIN_TYPE = None
DISTURBANCE_CONDITION = None

# If True, files that do not include all four RL2AC parameters in the filename are skipped.
# This is usually what you want for parameter-search analysis.
REQUIRE_RL2AC_PARAMS = True

# Optional output directory for result tables
RESULTS_DIR = Path(EXP_FOLDER) / "analysis_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Default target used for height tracking.
BASE_HEIGHT_TARGET = 0.30

# Optional constants used only for limit violation metrics.
# Expected shapes:
#   JOINT_LIMITS: [2, num_dofs], where row 0 is lower and row 1 is upper
#   JOINT_TORQUE_LIMITS: scalar or [num_dofs]
JOINT_LIMITS = np.array([
    [-1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721],
    [ 1.047,  2.966, -0.837,  1.047,  2.966, -0.837,  1.047,  2.966, -0.837,  1.047,  2.966, -0.837],
])
JOINT_TORQUE_LIMITS = np.array([23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55])

EPS = 1e-8
EXCLUDE_DIRS = ["analysis_results"]
NUM_WORKERS = 20

## Loading utilities

In [ ]:
def string_to_array(array_string):
    """Convert stringified list/array columns from the logger into Python lists."""
    if isinstance(array_string, str):
        try:
            return ast.literal_eval(array_string)
        except (ValueError, SyntaxError):
            return array_string
    return array_string


ARRAY_COLUMNS = [
    "base_cmd",
    "base_pose",
    "base_rpy",
    "dof_pose",
    "base_lin_vel",
    "base_ang_vel",
    "dof_vel",
    "proj_grav",
    "feet_pos",
    "tau_act",
    "grf",
    "q_des",
    "tau_ff",
    "tau_pd",
    "payload",
    "com_shift",
    "rand_push",
    "rand_wrench",
]


def get_csv_converters(columns=ARRAY_COLUMNS):
    return {col: string_to_array for col in columns}


PARAM_SUFFIX_RE = re.compile(
    r"(?:^|_)alpha_([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
    r"_kappa_([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
    r"_lambda_?0_([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
    r"_k_?0_([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)$"
)


def parse_rl2ac_eval_filename(path, approach):
    """
    Parse filenames of the form:
        {approach}_{terrain}_{disturbance_condition}_alpha_{alpha}_kappa_{kappa}_lambda0_{lambda_0}_k0_{k_0}.csv

    Notes:
        - `approach` may itself contain underscores, e.g. `go1_rl2ac`.
        - The terrain token is assumed to be the first token after the approach prefix.
        - Everything between terrain and the parameter suffix is treated as the disturbance condition.
    """
    path = Path(path)
    stem = path.stem

    prefix = f"{approach}_"
    if not stem.startswith(prefix):
        return None

    param_match = PARAM_SUFFIX_RE.search(stem)
    if param_match is None:
        if REQUIRE_RL2AC_PARAMS:
            return None
        alpha = kappa = lambda_0 = k_0 = np.nan
        no_param_stem = stem
    else:
        alpha, kappa, lambda_0, k_0 = [float(x) for x in param_match.groups()]
        no_param_stem = stem[:param_match.start()]
        if no_param_stem.endswith("_"):
            no_param_stem = no_param_stem[:-1]

    remainder = no_param_stem[len(prefix):]
    parts = remainder.split("_")

    if len(parts) < 2:
        return None

    terrain = parts[0]
    disturbance_condition = "_".join(parts[1:])

    return {
        "approach": approach,
        "terrain": terrain,
        "disturbance_condition": disturbance_condition,
        "alpha": alpha,
        "kappa": kappa,
        "lambda_0": lambda_0,
        "k_0": k_0,
        "path": path,
    }


def find_rl2ac_eval_files(exp_folder, approach, terrain_type=None, disturbance_condition=None):
    exp_folder = Path(exp_folder)

    records = []
    for path in sorted(exp_folder.rglob(f"{approach}_*.csv")):
        info = parse_rl2ac_eval_filename(path, approach)
        if info is None:
            continue

        # Skip blacklisted subdirectories
        if EXCLUDE_DIRS:
            rel_parts = path.relative_to(exp_folder).parts
            if any(part in EXCLUDE_DIRS for part in rel_parts):
                continue

        if terrain_type is not None and info["terrain"] != terrain_type:
            continue

        if disturbance_condition is not None and info["disturbance_condition"] != disturbance_condition:
            continue

        records.append(info)

    return pd.DataFrame(records)


def load_eval_csv(path):
    """Load one evaluation CSV with logger array columns converted."""
    return pd.read_csv(path, converters=get_csv_converters())


def _load_one_rl2ac_eval_file(row):
    df = load_eval_csv(row["path"])

    metadata_cols = [
        "approach",
        "terrain",
        "disturbance_condition",
        "alpha",
        "kappa",
        "lambda_0",
        "k_0",
    ]
    for col in metadata_cols:
        df[col] = row[col]

    df["source_file"] = str(row["path"])
    return df


def load_rl2ac_eval_dataset(file_table):
    dfs = [_load_one_rl2ac_eval_file(row) for _, row in file_table.iterrows()]
    if not dfs:
        raise FileNotFoundError("No matching RL2AC evaluation CSV files were found.")
    return pd.concat(dfs, ignore_index=True)


def load_rl2ac_eval_dataset_parallel(file_table, max_workers=10):
    rows = [row for _, row in file_table.iterrows()]
    if not rows:
        raise FileNotFoundError("No matching RL2AC evaluation CSV files were found.")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(executor.map(_load_one_rl2ac_eval_file, rows))

    return pd.concat(dfs, ignore_index=True)

In [ ]:
file_table = find_rl2ac_eval_files(
    EXP_FOLDER,
    APPROACH,
    terrain_type=TERRAIN_TYPE,
    disturbance_condition=DISTURBANCE_CONDITION,
)

print(f"Found {len(file_table)} parameter-evaluation CSV files")
file_table.head()

In [ ]:
sub_table_1 = file_table.iloc[0:25]
sub_table_2 = file_table.iloc[25:50]
sub_table_3 = file_table.iloc[50:75]
sub_table_4 = file_table.iloc[75:]

In [ ]:
df_all = load_rl2ac_eval_dataset_parallel(sub_table_1, max_workers=NUM_WORKERS)

print(f"Loaded {len(sub_table_1)} files")
print(f"Loaded {len(df_all):,} rows")
print("Terrains:", sorted(df_all["terrain"].unique()))
print("Disturbance conditions:", sorted(df_all["disturbance_condition"].unique()))
print("Parameter combinations:", df_all[["alpha", "kappa", "lambda_0", "k_0"]].drop_duplicates().shape[0])

df_all.head()

## Metric utilities

In [ ]:
def as_array(df, column, valid_mask=None):
    """Convert a dataframe column of list-like entries into a numpy array."""
    if column not in df.columns:
        return None

    series = df[column]
    if valid_mask is not None:
        series = series.loc[valid_mask]

    if len(series) == 0:
        return None

    return np.asarray(series.to_list())


def safe_mean(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanmean(x))


def safe_std(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanstd(x))


def rmse(x):
    x = np.asarray(x)
    return float(np.sqrt(np.nanmean(np.square(x))))


def mae(x):
    x = np.asarray(x)
    return float(np.nanmean(np.abs(x)))


def mean_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanmean(np.linalg.norm(x, axis=axis, ord=1)))


def std_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanstd(np.linalg.norm(x, axis=axis, ord=1)))


def rpy_to_rotmat(rpy):
    """
    rpy: (..., 3) roll, pitch, yaw
    returns: (..., 3, 3) rotation matrix (base -> world)
    """
    roll, pitch, yaw = rpy[..., 0], rpy[..., 1], rpy[..., 2]

    cr, sr = np.cos(roll),  np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw),   np.sin(yaw)

    R = np.stack([
        np.stack([cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr], axis=-1),
        np.stack([sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr], axis=-1),
        np.stack([-sp,   cp*sr,            cp*cr           ], axis=-1)
    ], axis=-2)

    return R

In [ ]:
def compute_eval_metrics(
    df,
    approach='pact',
    base_height_target=0.30,
    joint_limits=None,
    joint_torque_limits=None,
    pact_metric_approaches=("pact", "abl"),
    eps=1e-8,
):
    """
    Compute scalar evaluation metrics for one dataframe.

    Failed rows are excluded from continuous metrics but counted in failure statistics.
    """
    metrics = {}

    use_pact_metrics = (
        approach is not None
        and any(k.lower() in approach.lower() for k in pact_metric_approaches)
    )

    n_total = len(df)
    failure = df["failure"].astype(float).to_numpy() if "failure" in df.columns else np.zeros(n_total)
    valid_mask = failure == 0

    metrics["num_rows_total"] = int(n_total)
    metrics["num_rows_valid"] = int(valid_mask.sum())
    metrics["num_failures"] = int(failure.sum())
    metrics["failure_rate"] = float(failure.mean()) if n_total > 0 else np.nan

    if valid_mask.sum() == 0:
        return metrics

    q_actions = as_array(df, "q_des", valid_mask)
    q_obs = as_array(df, "dof_pose", valid_mask)

    vel_cmds = as_array(df, "base_cmd", valid_mask)
    lin_vel = as_array(df, "base_lin_vel", valid_mask)
    ang_vel = as_array(df, "base_ang_vel", valid_mask)

    base_pose = as_array(df, "base_pose", valid_mask)
    proj_grav = as_array(df, "proj_grav", valid_mask)

    q_vel = as_array(df, "dof_vel", valid_mask)
    q_tau = as_array(df, "tau_act", valid_mask)

    grfs = as_array(df, "grf", valid_mask)
    ff_tau = as_array(df, "tau_ff", valid_mask)
    pd_tau = as_array(df, "tau_pd", valid_mask)

    # Joint tracking
    if q_actions is not None and q_obs is not None:
        dof_errors = q_actions - q_obs
        metrics["dof_tracking_rmse"] = rmse(dof_errors)
        metrics["dof_tracking_mae"] = mae(dof_errors)
        metrics["q_action_norm_mean"] = mean_norm(q_actions)
        metrics["q_action_norm_std"] = std_norm(q_actions)

        if joint_limits is not None:
            joint_limits = np.asarray(joint_limits)
            joint_pred_limits_error = -(q_actions - joint_limits[0, :]).clip(max=0.0)
            joint_pred_limits_error += (q_actions - joint_limits[1, :]).clip(min=0.0)
            metrics["joint_limit_violation_mean"] = float(np.mean(joint_pred_limits_error))

    # Command tracking
    if vel_cmds is not None and lin_vel is not None and ang_vel is not None:
        lin_cmd_errors = vel_cmds[:, 0:2] - lin_vel[:, 0:2]
        ang_cmd_errors = vel_cmds[:, 2] - ang_vel[:, 2]
        cmd_errs = np.concatenate((lin_cmd_errors, ang_cmd_errors[:, None]), axis=1)

        metrics["lin_cmd_rmse"] = rmse(lin_cmd_errors)
        metrics["lin_cmd_mae"] = mae(lin_cmd_errors)
        metrics["lin_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(lin_cmd_errors))))

        metrics["ang_cmd_rmse"] = rmse(ang_cmd_errors)
        metrics["ang_cmd_mae"] = mae(ang_cmd_errors)
        metrics["ang_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(ang_cmd_errors))))

        metrics["total_cmd_rmse"] = rmse(cmd_errs)
        metrics["total_cmd_mae"] = mae(cmd_errs)

    # Height / orientation / unwanted velocity
    if base_pose is not None:
        height_errors = base_height_target - base_pose[:, 2]
        metrics["height_rmse"] = rmse(height_errors)
        metrics["height_mae"] = mae(height_errors)

    if proj_grav is not None:
        orientation_norm = np.linalg.norm(proj_grav[:, 0:2], axis=1, ord=1)
        metrics["projected_gravity_rp_norm_mean"] = safe_mean(orientation_norm)
        metrics["projected_gravity_rp_norm_std"] = safe_std(orientation_norm)

    if lin_vel is not None and ang_vel is not None:
        z_vel = lin_vel[:, 2]
        ang_vel_rp_norm = np.linalg.norm(ang_vel[:, 0:2], axis=1, ord=1)
        total_unwanted_vel = np.concatenate((z_vel[:, None], ang_vel[:, 0:2]), axis=1)
        total_unwanted_norm = np.linalg.norm(total_unwanted_vel, axis=1, ord=1)

        metrics["z_vel_rmse"] = rmse(z_vel)
        metrics["z_vel_mae"] = mae(z_vel)
        metrics["ang_vel_rp_norm_mean"] = safe_mean(ang_vel_rp_norm)
        metrics["ang_vel_rp_norm_std"] = safe_std(ang_vel_rp_norm)
        metrics["total_unwanted_vel_norm_mean"] = safe_mean(total_unwanted_norm)
        metrics["total_unwanted_vel_norm_std"] = safe_std(total_unwanted_norm)

    # Torque / force / power
    if q_vel is not None and q_tau is not None:
        joint_power = q_vel * q_tau
        metrics["joint_power_norm_mean"] = mean_norm(joint_power)
        metrics["joint_power_norm_std"] = std_norm(joint_power)

    if grfs is not None:
        # Handle either [N, 4, 3] or flattened [N, 12].
        grf_flat = grfs.reshape(grfs.shape[0], -1)
        metrics["grf_norm_mean"] = mean_norm(grf_flat)
        metrics["grf_norm_std"] = std_norm(grf_flat)

    if ff_tau is not None:
        metrics["ff_tau_norm_mean"] = mean_norm(ff_tau)
        metrics["ff_tau_norm_std"] = std_norm(ff_tau)

    if pd_tau is not None:
        metrics["pd_tau_norm_mean"] = mean_norm(pd_tau)
        metrics["pd_tau_norm_std"] = std_norm(pd_tau)

    if use_pact_metrics and ff_tau is not None and pd_tau is not None:
        total_tau_cmd = ff_tau + pd_tau
        metrics["total_tau_cmd_norm_mean"] = mean_norm(total_tau_cmd)
        metrics["total_tau_cmd_norm_std"] = std_norm(total_tau_cmd)

        ff_norm = np.linalg.norm(ff_tau, axis=1)
        pd_norm = np.linalg.norm(pd_tau, axis=1)
        metrics["ff_tau_ratio_mean"] = safe_mean(ff_norm / (ff_norm + pd_norm + eps))
        metrics["pd_tau_ratio_mean"] = safe_mean(pd_norm / (ff_norm + pd_norm + eps))
        metrics["pd_to_ff_tau_norm_ratio"] = float(np.mean(pd_norm) / (np.mean(ff_norm) + eps))

        if joint_torque_limits is not None:
            joint_torque_limits = np.asarray(joint_torque_limits)
            violation = -(total_tau_cmd - (-joint_torque_limits)).clip(max=0.0)
            violation += (total_tau_cmd - joint_torque_limits).clip(min=0.0)
            metrics["joint_torque_limit_violation_mean"] = float(np.mean(violation))

    # FF/PD power interaction metrics
    if use_pact_metrics and ff_tau is not None and pd_tau is not None and q_vel is not None:
        ff_power = ff_tau * q_vel
        pd_power = pd_tau * q_vel
        total_power = (ff_tau + pd_tau) * q_vel

        dot = np.sum(ff_power * pd_power, axis=1)
        ff_power_norm = np.linalg.norm(ff_power, axis=1)
        pd_power_norm = np.linalg.norm(pd_power, axis=1)

        cosine_sim = dot / ((ff_power_norm * pd_power_norm) + eps)

        metrics["ff_power_norm_mean"] = safe_mean(ff_power_norm)
        metrics["pd_power_norm_mean"] = safe_mean(pd_power_norm)
        metrics["pd_to_ff_power_ratio"] = float(np.mean(pd_power_norm) / (np.mean(ff_power_norm) + eps))
        metrics["power_alignment_mean"] = safe_mean(cosine_sim)
        metrics["power_alignment_std"] = safe_std(cosine_sim)

        neg_dot = np.maximum(-dot, 0.0)
        metrics["fraction_antagonistic_energy"] = float(np.sum(neg_dot) / (np.sum(np.abs(dot)) + eps))

        numerator = np.abs(ff_power) + np.abs(pd_power) - np.abs(total_power)
        denominator = np.abs(ff_power) + np.abs(pd_power)
        metrics["internal_power_cancellation"] = float(np.mean(numerator) / (np.mean(denominator) + eps))

    return metrics

## Compute per-parameter metrics

In [ ]:
PARAM_COLS = ["alpha", "kappa", "lambda_0", "k_0"]
BASE_GROUP_COLS = ["approach", "terrain", "disturbance_condition"] + PARAM_COLS


def summarize_by_parameter_and_terrain(df_all):
    rows = []

    for keys, df_group in df_all.groupby(BASE_GROUP_COLS, dropna=False):
        key_dict = dict(zip(BASE_GROUP_COLS, keys))
        approach = key_dict["approach"]

        metrics = compute_eval_metrics(
            df_group,
            approach=approach,
            pact_metric_approaches=("pact", "abl", "rl2ac"),
            base_height_target=BASE_HEIGHT_TARGET,
            joint_limits=JOINT_LIMITS,
            joint_torque_limits=JOINT_TORQUE_LIMITS,
            eps=EPS,
        )

        rows.append({**key_dict, **metrics})

    return pd.DataFrame(rows)


def summarize_by_parameter_all_terrains(df_all):
    rows = []

    group_cols = ["approach", "disturbance_condition"] + PARAM_COLS
    for keys, df_group in df_all.groupby(group_cols, dropna=False):
        key_dict = dict(zip(group_cols, keys))
        approach = key_dict["approach"]

        metrics = compute_eval_metrics(
            df_group,
            approach=approach,
            pact_metric_approaches=("pact", "abl", "rl2ac"),
            base_height_target=BASE_HEIGHT_TARGET,
            joint_limits=JOINT_LIMITS,
            joint_torque_limits=JOINT_TORQUE_LIMITS,
            eps=EPS,
        )

        rows.append({
            **key_dict,
            "terrain": "ALL",
            **metrics,
        })

    return pd.DataFrame(rows)


per_param_terrain_results = summarize_by_parameter_and_terrain(df_all)
per_param_all_terrain_results = summarize_by_parameter_all_terrains(df_all)
combined_param_results = pd.concat(
    [per_param_terrain_results, per_param_all_terrain_results],
    ignore_index=True,
)

# Useful stable ordering for inspection
sort_cols = ["disturbance_condition", "terrain", "alpha", "kappa", "lambda_0", "k_0"]
combined_param_results = combined_param_results.sort_values(sort_cols, ignore_index=True)

combined_param_results

In [ ]:
# Save result tables
prefix = f"{APPROACH}_rl2ac_param_sweep"

per_param_terrain_path = RESULTS_DIR / f"{prefix}_per_terrain_results.csv"
per_param_all_terrain_path = RESULTS_DIR / f"{prefix}_all_terrain_results.csv"
combined_param_path = RESULTS_DIR / f"{prefix}_combined_results.csv"

per_param_terrain_results.to_csv(per_param_terrain_path, index=False)
per_param_all_terrain_results.to_csv(per_param_all_terrain_path, index=False)
combined_param_results.to_csv(combined_param_path, index=False)

print("Saved:")
print(per_param_terrain_path)
print(per_param_all_terrain_path)
print(combined_param_path)

## Compact result views

In [ ]:
# Choose the metrics you care about most for a compact table.
summary_cols = [
    "approach",
    "alpha",
    "kappa",
    "lambda_0",
    "k_0",
    "num_failures",
    "height_mae",
    "lin_cmd_mae",
    "ang_cmd_mae",
    "projected_gravity_rp_norm_mean",
    "z_vel_mae",
    "ang_vel_rp_norm_mean",
    "dof_tracking_mae"
]

available_summary_cols = [c for c in summary_cols if c in combined_param_results.columns]
compact_param_results = combined_param_results[available_summary_cols].copy()
compact_param_results

In [ ]:
compact_param_results.round(4)

## Rank parameter combinations

In [ ]:
# Rank on ALL-terrain rows by a selected metric.
# Lower is better for the default tracking/error metrics.
RANK_METRIC = "total_cmd_rmse"
RANK_TERRAIN = "ALL"
RANK_DISTURBANCE_CONDITION = None  # e.g., "payload" or None for all conditions

rank_df = combined_param_results.copy()
rank_df = rank_df[rank_df["terrain"] == RANK_TERRAIN]

if RANK_DISTURBANCE_CONDITION is not None:
    rank_df = rank_df[rank_df["disturbance_condition"] == RANK_DISTURBANCE_CONDITION]

rank_cols = [
    "approach",
    "terrain",
    "disturbance_condition",
    "alpha",
    "kappa",
    "lambda_0",
    "k_0",
    "num_failures",
    "failure_rate",
    RANK_METRIC,
]
rank_cols = [c for c in rank_cols if c in rank_df.columns]

ranked_param_results = rank_df[rank_cols].sort_values(RANK_METRIC, ascending=True).reset_index(drop=True)
ranked_param_results.head(20)

## Plot selected metrics over parameter values

In [ ]:
def plot_metric_vs_param(results, metric, param, terrain="ALL", disturbance_condition=None):
    plot_df = results.copy()
    plot_df = plot_df[plot_df["terrain"] == terrain]

    if disturbance_condition is not None:
        plot_df = plot_df[plot_df["disturbance_condition"] == disturbance_condition]

    if metric not in plot_df.columns:
        raise KeyError(f"{metric} not found in results.")
    if param not in plot_df.columns:
        raise KeyError(f"{param} not found in results.")

    plt.figure(figsize=(7, 4))
    if sns is not None:
        sns.scatterplot(data=plot_df, x=param, y=metric, hue="disturbance_condition")
    else:
        for condition, group in plot_df.groupby("disturbance_condition"):
            plt.scatter(group[param], group[metric], label=condition)
        plt.legend()

    plt.title(f"{metric} vs {param} ({terrain})")
    plt.tight_layout()
    plt.show()


for param in PARAM_COLS:
    if "total_cmd_rmse" in combined_param_results.columns:
        plot_metric_vs_param(combined_param_results, "total_cmd_rmse", param, terrain="ALL")

## Optional: parameter heatmaps

In [ ]:
def plot_param_heatmap(
    results,
    metric,
    x_param="alpha",
    y_param="kappa",
    terrain="ALL",
    disturbance_condition=None,
    fixed_params=None,
    aggfunc="mean",
):
    """
    Heatmap over two parameters. For LHS/random searches, cells may be sparse;
    this is most useful for grid or staged one-factor sweeps.
    """
    plot_df = results.copy()
    plot_df = plot_df[plot_df["terrain"] == terrain]

    if disturbance_condition is not None:
        plot_df = plot_df[plot_df["disturbance_condition"] == disturbance_condition]

    if fixed_params:
        for name, value in fixed_params.items():
            plot_df = plot_df[np.isclose(plot_df[name], value)]

    pivot = plot_df.pivot_table(
        index=y_param,
        columns=x_param,
        values=metric,
        aggfunc=aggfunc,
    )

    plt.figure(figsize=(8, 5))
    if sns is not None:
        sns.heatmap(pivot, annot=True, fmt=".3g")
    else:
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar(label=metric)
        plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
        plt.yticks(range(len(pivot.index)), pivot.index)

    plt.title(f"{metric}: {y_param} vs {x_param}")
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.tight_layout()
    plt.show()


# Example: uncomment after running a grid-style sweep.
# plot_param_heatmap(
#     combined_param_results,
#     metric="total_cmd_rmse",
#     x_param="alpha",
#     y_param="kappa",
#     terrain="ALL",
#     fixed_params={"lambda_0": 0.3, "k_0": 5.0},
# )